

📝 Text Summarization

## Text and Speech Analysis – Mini Project 4

This project implements an extractive text summarization system
using Natural Language Processing techniques.


In [1]:
!pip install -q nltk gradio

In [2]:
import nltk
import re
import pandas as pd
import gradio as gr

from collections import Counter

In [3]:
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [4]:
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

In [6]:
text = """
Artificial intelligence is transforming many industries around the world.
Healthcare organizations use artificial intelligence for disease detection,
medical imaging, and patient monitoring.
Banks use artificial intelligence to detect fraudulent transactions and
improve financial security.
Manufacturing companies use AI for predictive maintenance and automation.
In education, artificial intelligence can personalize learning experiences
for students.
AI-powered systems can analyze large amounts of information much faster
than humans.
As artificial intelligence continues to develop, organizations are
increasingly adopting intelligent technologies to improve efficiency,
reduce costs, and support decision making.
However, organizations must also consider privacy, security, and ethical
issues when implementing artificial intelligence systems.
"""

In [7]:
sentences = sent_tokenize(text)

print("Number of sentences:", len(sentences))

for i, sentence in enumerate(sentences, 1):
    print(i, sentence)

Number of sentences: 8
1 
Artificial intelligence is transforming many industries around the world.
2 Healthcare organizations use artificial intelligence for disease detection,
medical imaging, and patient monitoring.
3 Banks use artificial intelligence to detect fraudulent transactions and
improve financial security.
4 Manufacturing companies use AI for predictive maintenance and automation.
5 In education, artificial intelligence can personalize learning experiences
for students.
6 AI-powered systems can analyze large amounts of information much faster
than humans.
7 As artificial intelligence continues to develop, organizations are
increasingly adopting intelligent technologies to improve efficiency,
reduce costs, and support decision making.
8 However, organizations must also consider privacy, security, and ethical
issues when implementing artificial intelligence systems.


In [8]:
stop_words = set(stopwords.words("english"))

print(list(stop_words)[:20])

['a', 'him', "i've", 'hasn', 'below', "they'd", 'what', 'but', 'having', 'both', "you'd", 'should', 'between', 'won', "he'll", 'about', 'such', 'or', 'me', "she'd"]


In [9]:
words = word_tokenize(text.lower())

word_frequencies = Counter()

for word in words:

    if word.isalpha() and word not in stop_words:
        word_frequencies[word] += 1

print(word_frequencies.most_common(10))

[('artificial', 6), ('intelligence', 6), ('organizations', 3), ('use', 3), ('improve', 2), ('security', 2), ('systems', 2), ('transforming', 1), ('many', 1), ('industries', 1)]


In [10]:
max_frequency = max(word_frequencies.values())

for word in word_frequencies:
    word_frequencies[word] = (
        word_frequencies[word] / max_frequency
    )

In [11]:
sentence_scores = {}

for sentence in sentences:

    sentence_words = word_tokenize(sentence.lower())

    score = 0

    for word in sentence_words:

        if word in word_frequencies:
            score += word_frequencies[word]

    sentence_scores[sentence] = score

In [12]:
for sentence, score in sentence_scores.items():

    print(round(score, 3), "→", sentence)

2.833 → 
Artificial intelligence is transforming many industries around the world.
4.167 → Healthcare organizations use artificial intelligence for disease detection,
medical imaging, and patient monitoring.
4.0 → Banks use artificial intelligence to detect fraudulent transactions and
improve financial security.
1.5 → Manufacturing companies use AI for predictive maintenance and automation.
2.833 → In education, artificial intelligence can personalize learning experiences
for students.
1.5 → AI-powered systems can analyze large amounts of information much faster
than humans.
4.833 → As artificial intelligence continues to develop, organizations are
increasingly adopting intelligent technologies to improve efficiency,
reduce costs, and support decision making.
4.5 → However, organizations must also consider privacy, security, and ethical
issues when implementing artificial intelligence systems.


In [13]:
def summarize_text(text, summary_ratio=0.3):

    sentences = sent_tokenize(text)

    if len(sentences) <= 2:
        return text

    words = word_tokenize(text.lower())

    stop_words = set(stopwords.words("english"))

    word_frequencies = Counter()

    for word in words:

        if word.isalpha() and word not in stop_words:
            word_frequencies[word] += 1

    if not word_frequencies:
        return text

    max_frequency = max(word_frequencies.values())

    for word in word_frequencies:
        word_frequencies[word] /= max_frequency

    sentence_scores = {}

    for sentence in sentences:

        sentence_words = word_tokenize(sentence.lower())

        score = 0

        for word in sentence_words:

            if word in word_frequencies:
                score += word_frequencies[word]

        sentence_scores[sentence] = score

    number_of_sentences = max(
        1,
        int(len(sentences) * summary_ratio)
    )

    ranked_sentences = sorted(
        sentence_scores,
        key=sentence_scores.get,
        reverse=True
    )

    selected = ranked_sentences[:number_of_sentences]

    # Restore original article order
    summary = [
        sentence for sentence in sentences
        if sentence in selected
    ]

    return " ".join(summary)

In [14]:
summary = summarize_text(text)

print("SUMMARY:\n")
print(summary)

SUMMARY:

As artificial intelligence continues to develop, organizations are
increasingly adopting intelligent technologies to improve efficiency,
reduce costs, and support decision making. However, organizations must also consider privacy, security, and ethical
issues when implementing artificial intelligence systems.


In [15]:
short_summary = summarize_text(
    text,
    summary_ratio=0.2
)

print(short_summary)

As artificial intelligence continues to develop, organizations are
increasingly adopting intelligent technologies to improve efficiency,
reduce costs, and support decision making.


In [16]:
medium_summary = summarize_text(
    text,
    summary_ratio=0.4
)

print(medium_summary)

Healthcare organizations use artificial intelligence for disease detection,
medical imaging, and patient monitoring. As artificial intelligence continues to develop, organizations are
increasingly adopting intelligent technologies to improve efficiency,
reduce costs, and support decision making. However, organizations must also consider privacy, security, and ethical
issues when implementing artificial intelligence systems.


In [17]:
long_summary = summarize_text(
    text,
    summary_ratio=0.6
)

print(long_summary)

Healthcare organizations use artificial intelligence for disease detection,
medical imaging, and patient monitoring. Banks use artificial intelligence to detect fraudulent transactions and
improve financial security. As artificial intelligence continues to develop, organizations are
increasingly adopting intelligent technologies to improve efficiency,
reduce costs, and support decision making. However, organizations must also consider privacy, security, and ethical
issues when implementing artificial intelligence systems.


In [18]:
original_words = len(word_tokenize(text))

summary_words = len(
    word_tokenize(summary)
)

print("Original word count:", original_words)
print("Summary word count:", summary_words)

reduction = (
    1 - summary_words / original_words
) * 100

print(
    "Text reduction:",
    round(reduction, 2),
    "%"
)

Original word count: 118
Summary word count: 44
Text reduction: 62.71 %


In [19]:
def summarizer_app(text, ratio):

    if not text.strip():
        return "Please enter some text."

    summary = summarize_text(
        text,
        float(ratio)
    )

    return summary

In [20]:
with gr.Blocks(title="Text Summarization") as app:

    gr.Markdown("# 📝 Text Summarization")

    gr.Markdown(
        "Enter a long text and generate an extractive summary."
    )

    text_input = gr.Textbox(
        label="Enter Text",
        lines=10,
        placeholder="Paste your article or paragraph here..."
    )

    ratio = gr.Slider(
        minimum=0.1,
        maximum=0.8,
        value=0.3,
        step=0.1,
        label="Summary Length"
    )

    summarize_button = gr.Button(
        "Generate Summary"
    )

    output = gr.Textbox(
        label="Generated Summary",
        lines=8
    )

    summarize_button.click(
        summarizer_app,
        inputs=[text_input, ratio],
        outputs=output
    )

app.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b2937ee216a6b1eca8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
